# Promotion uplift - baseline method

Sprint 9. Used to agree the uplift definition with the commercial team on
12/03: baseline is the median of the four non-promo weeks before the offer,
uplift is the ratio of promo week units to that baseline.

The pipeline has since moved to the lag/rolling block in
`feature_engineering`, but the definition below is still the one in the
steerco pack. -- MK

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

from meridian.utils import nordfalk_codes as codes
from meridian.utils import fiscal_calendar as fc

%matplotlib inline

In [ ]:
from kedro.framework.context import load_context

context = load_context('../')
catalog = context.catalog
primary = catalog.load('demand_primary')
primary.shape

In [ ]:
# Promo weeks per mechanic. AVIS is the leaflet, TILBOD the shelf offer.
primary.groupby('promo_code')['units'].agg(['count', 'mean'])

In [ ]:
def baseline_units(frame, weeks=4):
    frame = frame.sort_values('week_label')
    base = frame[frame['promo_flag'] == 0]['units'].rolling(weeks, min_periods=2).median()
    frame['baseline_units'] = base.shift(1)
    return frame

panel = primary.groupby(['store_id', 'sku_id']).apply(baseline_units)
panel = panel.reset_index(drop=True)
panel['uplift'] = panel['units'] / panel['baseline_units']
panel[panel['promo_flag'] == 1]['uplift'].describe()

In [ ]:
uplift_by_cat = (
    panel[panel['promo_flag'] == 1]
    .groupby('category_code')['uplift']
    .median()
    .sort_values(ascending=False)
)
uplift_by_cat.index = [codes.category_name(c) for c in uplift_by_cat.index]
uplift_by_cat.plot(kind='barh', figsize=(8, 5))
plt.xlabel('median uplift x baseline')
plt.show()

Leaflet pages 1-3 behave differently enough that the commercial team wanted
them split out. Not in the model yet.

In [ ]:
leaflet = panel[(panel['promo_flag'] == 1) & (panel['is_leaflet'] == 1)].copy()
leaflet['front_page'] = (leaflet['leaflet_page'] <= 3).astype(int)
leaflet.groupby('front_page')['uplift'].median()

In [ ]:
# Log-log elasticity on the promo weeks only, as a cross check on the
# pipeline's elasticity model.
fit_frame = panel.dropna(subset=['unit_price_dkk', 'baseline_units'])
fit_frame = fit_frame[(fit_frame['units'] > 0) & (fit_frame['unit_price_dkk'] > 0)]
X = sm.add_constant(np.log(fit_frame['unit_price_dkk']))
model = sm.OLS(np.log(fit_frame['units']), X).fit()
model.params

In [ ]:
# Fresh categories are much less elastic than the dry grocery lines.
for code in codes.FRESH_CATEGORY_CODES:
    subset = fit_frame[fit_frame['category_code'] == code]
    if len(subset) < 50:
        continue
    Xc = sm.add_constant(np.log(subset['unit_price_dkk']))
    fit = sm.OLS(np.log(subset['units']), Xc).fit()
    print(codes.category_name(code), round(fit.params[1], 3), len(subset))

TODO(MK, sprint 11): rerun once the price history is loaded at store level.
The version above uses the realised unit price, which already carries the
promotion, so the elasticity is biased towards zero.